# Generate And Visualize Mock EGM Data

This notebook uses `generate_mock_data.py` to create synthetic EGM arrays in the project `data/` directory and visualizes sample waveforms and labels.

In [1]:
from pathlib import Path
import sys
import numpy as np
import matplotlib.pyplot as plt

# Resolve project root from this notebook location
project_root = Path.cwd().resolve().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from generate_mock_data import generate_mock_egm_signals, generate_mock_labels, save_mock_data

print(f'Project root: {project_root}')

Project root: /Users/xichongliu/Desktop/PGY-3/VT-EGM manuscript.nosync/code/egm-to-scar-mapping


In [ ]:
# Config
n_samples = 120
n_channels = 3
n_timesteps = 2500
random_seed = 42
output_dir = project_root / 'data' / 'sample_data'

print(f'Output directory: {output_dir}')

In [ ]:
# Generate mock arrays
signals = generate_mock_egm_signals(
    n_samples=n_samples,
    n_channels=n_channels,
    n_timesteps=n_timesteps,
    random_state=random_seed,
)
labels = generate_mock_labels(
    n_samples=n_samples,
    n_classes=3,
    random_state=random_seed,
)

save_mock_data(str(output_dir), signals, labels)

print('Saved files:')
print(' -', output_dir / 'egm_signals.npy')
print(' -', output_dir / 'egm_labels.npy')
print('Shapes:', signals.shape, labels.shape)

In [ ]:
# Reload from disk to verify persisted data
signals_disk = np.load(output_dir / 'egm_signals.npy')
labels_disk = np.load(output_dir / 'egm_labels.npy')

print('Loaded from disk:')
print('signals:', signals_disk.shape, signals_disk.dtype)
print('labels :', labels_disk.shape, labels_disk.dtype)
print('signal range:', float(signals_disk.min()), float(signals_disk.max()))

In [ ]:
# Plot a few examples to inspect ECG-like morphology
sampling_rate = 1000.0
time_sec = np.arange(signals_disk.shape[-1]) / sampling_rate
channel_names = ['Unipolar', 'Bipolar', 'Reference']
example_indices = [0, 1, 2]

fig, axes = plt.subplots(len(example_indices), 1, figsize=(14, 9), sharex=True)

for row, idx in enumerate(example_indices):
    ax = axes[row]
    for ch in range(signals_disk.shape[1]):
        ax.plot(time_sec, signals_disk[idx, ch], linewidth=1.0, label=channel_names[ch])

    ax.set_title(f'Sample {idx}  |  Labels: {labels_disk[idx].tolist()}')
    ax.set_ylabel('Amplitude (a.u.)')
    ax.grid(alpha=0.25)

axes[-1].set_xlabel('Time (seconds)')
axes[0].legend(loc='upper right', ncol=3)
plt.tight_layout()
plt.show()

In [ ]:
# Label distribution and scar burden summary
class_names = ['Endocardial', 'Mid-myocardial', 'Epicardial']
class_counts = labels_disk.sum(axis=0)
samples_with_any_scar = int((labels_disk.sum(axis=1) > 0).sum())

fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(class_names, class_counts, color=['#4C78A8', '#F58518', '#54A24B'])
ax.set_title('Label Counts Per Scar Class')
ax.set_ylabel('Count')
ax.grid(axis='y', alpha=0.25)
plt.tight_layout()
plt.show()

print(f'Samples with any scar: {samples_with_any_scar} / {labels_disk.shape[0]}')